# 1. Context

This notebook analyzes OCR performance of Tesseract over synthetic generated PDF images across variois degradation levels

# 2. Imports

In [ ]:
import pandas as pd
from pathlib import Path
from collections import defaultdict

In [85]:
import plotly.graph_objects as go
fig = go.Figure()
from plotly.subplots import make_subplots

# 2. Extracted Results MetaData

In [ ]:
writing_system_to_language = {'Devanagari': ['hindi','sanskrit','nepali','konkani'],
                              'tamil': ['tamil'],'telugu': ['telugu'],'Kannada': ['kannada'],'Malayalam': ['malayalam'],
                              'Bengali': ['bengali', 'assamese', 'manipuri'],'Meetei-mayek': ['manipuri'],'Gujarati': ['gujarati'],
                              'Gurmukhi': ['punjabi'],'Odia': ['oriya'],'Arabic': ['kashmiri', 'sindhi', 'urdu'],
                              'Latin': ['english'],'Ol-chiki': ['santali']}

## 2.1. Language Results Available

In [18]:
results_root = Path("../results/tesseract")

In [32]:
results_langs = [x.name.lower() for x in results_root.glob("*") if x.is_dir()]

In [33]:
language_to_writing_system = {
    "marathi": ["Devanagari"], "hindi": ["Devanagari"], "sanskrit": ["Devanagari"],
    "tamil": ["tamil"], "telugu": ["telugu"], "kannada": ["Kannada"],"malayalam": ["Malayalam"],
    "bengali": ["Bengali"], "assamese": ["Bengali"],"manipuri": ["Bengali", "Meetei-mayek"],"nepali": ["Devanagari"],
    "gujarati": ["Gujarati"], "punjabi": ["Gurmukhi"], "konkani": ["Devanagari"],"oriya": ["Odia"],"kashmiri": ["Devanagari", "Arabic"], 
    "sindhi": ["Arabic", "Devanagari"], "urdu": ["Arabic"],"english": ["Latin"], "santali": ["Ol-chiki", "Devanagari"],
    }

In [51]:
writing_sys_tessract_dict = defaultdict(list)

for language_res in results_langs:
    script = language_to_writing_system.get(language_res)[0]
    writing_sys_tessract_dict[script].append(language_res)

In [59]:
script_language_result = pd.Series(writing_sys_tessract_dict).to_frame(name='languages')
script_language_result.index.name = 'script'

In [78]:
def get_language_results_path(script: str, script_language_result: pd.DataFrame) -> list[Path]:
    """Get list of results path for given script"""

    results_script_lang = script_language_result.loc[script].to_list()[0]
    results_script_lang_path = [results_root.joinpath(lang).joinpath('results.csv') for lang in results_script_lang]

    return results_script_lang_path


In [88]:
script_language_result

,languages
script,
Gujarati,[gujarati]
Gurmukhi,[punjabi]
Devanagari,"[hindi, sanskrit, nepali]"
telugu,[telugu]
Bengali,"[assamese, bengali]"
Latin,[english]
Kannada,[kannada]
tamil,[tamil]


# 3. Results Across Various Writing System 

## 3.1. Devanagari

In [83]:
script = 'Devanagari'
results_devanagari_path = get_language_results_path(script, script_language_result)

In [81]:
results_list = []
for result_path in results_devanagari_path:
    lang = result_path.parent.name
    df_res = pd.read_csv(result_path)
    df_res['language'] = lang
    results_list.append(df_res)

results_devanagari = pd.concat(results_list)

In [84]:
lang_list = list(results_devanagari['language'].unique())

fig = make_subplots(rows=results_devanagari['language'].nunique(), 
                    cols=1, 
                    shared_yaxes=True,
                    subplot_titles=lang_list)

degrad_level_cer = ['cer_l0', 'cer_l1', 'cer_l2','cer_l3']

for idx, language in enumerate(lang_list):
    results_lang = results_devanagari.loc[results_devanagari['language'] == language]
    for cer_level in degrad_level_cer:
        fig.add_trace(go.Box(y=results_lang[cer_level] * 100,
                                name=cer_level,
                                text=results_lang['file_id']), 
                                row=idx + 1, col=1)


fig.update_layout(height=1200, width=1400)
fig.update_layout(showlegend=False, title_text=f"CER % across various degradation level in {script} languages")
fig.update_yaxes(title_text="CER in %")
fig.show()


In [86]:
lang_list = list(results_devanagari['language'].unique())

fig = make_subplots(rows=results_devanagari['language'].nunique(), 
                    cols=1, 
                    shared_yaxes=True,
                    subplot_titles=lang_list)

degrad_level_wer = ['wer_l0', 'wer_l1', 'wer_l2','wer_l3']

for idx, language in enumerate(lang_list):
    results_lang = results_devanagari.loc[results_devanagari['language'] == language]
    for wer_level in degrad_level_wer:
        fig.add_trace(go.Box(y=results_lang[wer_level] * 100,
                                name=wer_level,
                                text=results_lang['file_id']), 
                                row=idx + 1, col=1)


fig.update_layout(height=1200, width=1400)
fig.update_layout(showlegend=False, title_text=f"WER % across various degradation level and {script} languages")
fig.update_yaxes(title_text="WER in %")
fig.show()

In [87]:
results_lang

,file_id,ground_truth,ocr_output_L_0,ocr_output_L_1,ocr_output_L_2,ocr_output_L_3,cer_l0,cer_l1,cer_l2,cer_l3,wer_l0,wer_l1,wer_l2,wer_l3,language
0,Devanagari_Poppins_24,रूपमा यो राष्ट्रिय पनि भित्र सयौँ थुङ्गा फूलका...,"रुपमा यो राष्ट्रिय पनि भित्र थि ] का हामी मी,...",रुपमा यो राष्ट्रिय पनि भित्र दि ] का द्वामी ह...,रुपमा यो राष्ट्रिय पनि भित्र नेपालको राष्ट्रि...,"रुपमा यो राष्ट्रिय पनि भित्र न कामी मी, नेपाल...",0.636898,0.693890,0.743243,0.661575,0.798450,0.841085,0.895349,0.841085,nepali
1,Devanagari_Noto_Sans_5,"ट्रिनिडाड वैदिक, श्रुति पुरानो संस्कृतमा हिन्द...","ट्रिनिडाड वैदिक, श्रुति पुरानो संस्कृतमा हिन्...","ट्रिनिडाड वैदिक, श्रुति पुरानो संस्कृतमा हिन्...","ट्रिनिडाड वैदिक, श्रुति पुरानो संस्कृतमा एक ज...",' गति पुरानो संस्कृतमा ' जसलाई प्रायः धार्मिक...,0.287570,0.188312,0.234230,0.460575,0.445161,0.325806,0.377419,0.632258,nepali
2,Devanagari_Noto_Sans_Devanagari_21,ठुलो लगभग वर्षाको वर्षाको र वर्षा वायुमण्डलमा ...,ठुलो लगभग वर्षाको वर्षाको र वर्षा वायुमण्डलमा...,ठुलो लग भग वर्षाको वर्षाको र वर्षा वायुमण्डलम...,ठुलो लग भग वर्षाको वर्षाको र वर्षा वायुमण्डलम...,लौ लगभग वर्षाको वर्षाको र अत्यधिक गह्रौँ बा थ...,0.095977,0.103448,0.258621,0.478161,0.219697,0.200758,0.337121,0.647727,nepali
3,Devanagari_Noto_Sans_14,"एपल, आई.एस.ओ/आई.ई.सी. को सहायक होने याहा कोहि ...","एपल, आई.एस.ओ/आई.ई.सी. को सहायक होने याहा को [...","एपल, आई.एस.ओ/आई.ई.सी. को सहायक होने ना 1 कोहि...","एपल, आई.एस.ओ।आई.ई.सी. को सहायक होने गो (॥0८०५...","एपल, आई.एस.ओ/आई.ई.सी. को सहायक होने सा 1 कोहि...",0.200722,0.201238,0.185759,0.327657,0.322581,0.312903,0.267742,0.454839,nepali
4,Devanagari_Noto_Serif_Devanagari_1,"र कडा ""नीलो सूर्यको जीवित पृथ्वी सूर्यको दूरी ...","र कडा ""नीलो सूर्यको जीवित पृथ्वी सूर्यको दूरी...","र कडा ""नीलो सूर्यको जीवित पृथ्वी.सूर्यको दूरी...",पृथ्वी सूर्यको दूरी अनुसार सौर्यमण्ड्लमा रहेका...,"र कडा ""नीलो सूर्यको जीवित पृथ्वी सूर्यको दूरी...",0.231347,0.254382,0.496745,0.237356,0.336601,0.343137,0.689542,0.362745,nepali
5,Devanagari_Poppins_29,देखिन्छ। यो २८२२ प्रयोग वास्तविक इलेक्ट्रोनिक ...,देखिन्छ। यो 2822 प्रयोग वास्तविक हलेक्ट्रोनिक...,देखिन्छ। यो 2822 प्रयोग वास्तविक हलेक्ट्रोनिक...,देखिन्छ। यो 2822 प्रयोग वास्तविक इलेक्ट्रोनिक...,देखिन्छ। यो 2822 प्रयोग वास्तविक हुँनैडसि जि ...,0.482893,0.498445,0.527216,0.684292,0.718750,0.708333,0.755208,0.885417,nepali
6,Devanagari_Noto_Sans_18,गरे। ५ चन्द्रमा उनिहरूको निर्माण थोमस जेफरी ह्...,गरे। ५ चन्द्रमा उनिहरूको निर्माण थोमस जेफरी ह...,गरे। ५ चन्द्रमा उनिहरूको निर्माण थोमस जेफरी ह...,गरे। ५ चन्द्रमा उनिहरूको निर्माण थोमस जेफरी ह...,गरे। ५ चन्द्रमा उनिहरूको निर्माण जेफरी ह्याङ्...,0.121487,0.227108,0.234814,0.256573,0.266881,0.392283,0.369775,0.440514,nepali
7,Devanagari_Poppins_25,रूपमा भनेको धमिलो नाटकको र साहित्य भन्नाले सम्...,रुपमा भनेको धमिलो नाटकको र साहित्य भन्नाले सम...,रुपमा भनेको धमिलो नाटकको र साहित्य भन्नाले सम...,रुपमा भनेको धमिलो नाटकको र ल्ठिखित डि उ म ता ...,रुपमा भनेको धमिलो नाटकको र साहित्य भन्नाले अम...,0.498711,0.545747,0.578608,0.657861,0.686957,0.721739,0.765217,0.817391,nepali
8,Devanagari_Noto_Serif_Devanagari_23,हुन्छ। सक्छ हुन बगैंचा पनि पुस्तक वा किताब शव्...,हुन्छ। सक्छ हुन बगैंचा पनि पुस्तक वा किताब शव...,हुन्छ। सक्छ हुन बगैंचा पनि पुस्तक वा किताब शव...,हुन्छ। सक्छ हुन बगैंचा पनि पुस्तक वा किताब शव...,हुन्छ। सक्छ हुन बगैंचा पनि पुस्तक वा किताब शब...,0.159743,0.126390,0.209479,0.286132,0.297297,0.235521,0.351351,0.440154,nepali
9,Devanagari_Poppins_22,"भिसिआरबाट रे हो, सामान्यतया एक टेलिभिजन (अङ्ग्...","भिसिआरबाट रे हो, सामान्यतया एक ठेलिभिजन (भङ्ग...","भिसिआरबाट रे हो, सामान्यतया एक क (भङ्ग्रेण (2...","भिसिआरबाट रे हो, सामान्यतया एक पर्दा भएको एक ...","भिसिआरबाट रे हो, सामान्यतया एक अखिल त छोटकरीम...",0.553714,0.657714,0.688571,0.754286,0.743295,0.800766,0.812261,0.869732,nepali
